# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [1]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path(r"C:\Users\Li Xiaofeng\Downloads\E Commerce Dataset.xlsx"),
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")
print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
display(raw_df.head())


原始数据：C:\Users\Li Xiaofeng\Downloads\E Commerce Dataset.xlsx
项目输出目录：c:\Users\Li Xiaofeng\Documents\muc-commerce-2-24012416\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [2]:
# 任务1：确认项目对象
# 1. 每条记录代表一名电商用户的用户级行为与状态记录，而不是一笔订单。
# 2. 项目的目标变量是 Churn，1表示用户已流失，0表示用户未流失。
# 3. CustomerID只是用户的唯一标识，不具有连续数值的业务含义，因此不能求平均或用于大小比较。
print("任务1完成：已明确数据粒度、目标变量和CustomerID的使用边界。")


任务1完成：已明确数据粒度、目标变量和CustomerID的使用边界。


---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [3]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    return pd.DataFrame({
        "数据类型": data.dtypes.astype(str),
        "缺失数量": data.isna().sum(),
        "缺失比例(%)": data.isna().mean().mul(100),
        "唯一值数量": data.nunique(dropna=True),
    })

quality_before = build_quality_report(raw_df)
display(quality_before)


,数据类型,缺失数量,缺失比例(%),唯一值数量
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,object,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,object,0,0.00,7
Gender,object,0,0.00,2
HourSpendOnApp,float64,255,4.53,6
NumberOfDeviceRegistered,int64,0,0.00,6


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [4]:
# 任务2：项目初始审计
print("完全重复行数：", int(raw_df.duplicated().sum()))
print("CustomerID 重复数量：", int(raw_df["CustomerID"].duplicated().sum()))
print("Churn频数：")
print(raw_df["Churn"].value_counts().sort_index())
print("流失率：", f"{raw_df['Churn'].mean():.2%}")

for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts(dropna=False))


完全重复行数： 0
CustomerID 重复数量： 0
Churn频数：
Churn
0    4682
1     948
Name: count, dtype: int64
流失率： 16.84%

PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [5]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [6]:
def clean_ecommerce_data(data):
    """清洗电商用户行为数据，返回清洗结果与处理日志。"""
    cleaned_df = data.copy(deep=True)
    logs = []

    def add_log(step, rule, before, after, affected):
        logs.append({
            "处理步骤": step, "处理规则": rule,
            "处理前记录数": int(before), "处理后记录数": int(after),
            "影响记录数": int(affected)
        })

    before = len(cleaned_df)
    duplicate_count = int(cleaned_df.duplicated().sum())
    cleaned_df = cleaned_df.drop_duplicates().copy()
    add_log("删除完全重复行", "drop_duplicates", before, len(cleaned_df), duplicate_count)

    for col in NUMERIC_MISSING_COLS:
        affected = int(cleaned_df[col].isna().sum())
        median_value = cleaned_df[col].median()
        cleaned_df[col] = cleaned_df[col].fillna(median_value)
        add_log(f"填补{col}缺失值", f"总体中位数={median_value}", len(cleaned_df), len(cleaned_df), affected)

    for col, mappings in CATEGORY_MAPPINGS.items():
        for old_value, new_value in mappings.items():
            affected = int(cleaned_df[col].eq(old_value).sum())
            cleaned_df[col] = cleaned_df[col].replace(old_value, new_value)
            add_log(f"标准化{col}", f"{old_value} → {new_value}", len(cleaned_df), len(cleaned_df), affected)

    for col in ["Churn", "Complain"]:
        cleaned_df[col] = cleaned_df[col].astype(int)
        add_log(f"转换{col}类型", "转换为int", len(cleaned_df), len(cleaned_df), len(cleaned_df))

    return cleaned_df, pd.DataFrame(logs)


### 任务 3：运行清洗函数并查看日志

In [7]:
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)
display(cleaning_log)
display(cleaned_df.head())
assert raw_df is not cleaned_df
print("清洗函数执行完成，原始DataFrame未被覆盖。")


,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,删除完全重复行,drop_duplicates,5630,5630,0
1,填补Tenure缺失值,总体中位数=9.0,5630,5630,264
2,填补WarehouseToHome缺失值,总体中位数=14.0,5630,5630,251
3,填补HourSpendOnApp缺失值,总体中位数=3.0,5630,5630,255
4,填补OrderAmountHikeFromlastYear缺失值,总体中位数=15.0,5630,5630,265
5,填补CouponUsed缺失值,总体中位数=1.0,5630,5630,256
6,填补OrderCount缺失值,总体中位数=2.0,5630,5630,258
7,填补DaySinceLastOrder缺失值,总体中位数=3.0,5630,5630,307
8,标准化PreferredLoginDevice,Phone → Mobile Phone,5630,5630,1231
9,标准化PreferredPaymentMode,COD → Cash on Delivery,5630,5630,365


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


清洗函数执行完成，原始DataFrame未被覆盖。


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [8]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return {"Q1": q1, "Q3": q3, "下限": lower, "上限": upper,
            "候选异常值数量": int(((series < lower) | (series > upper)).sum())}

tenure_bins = [-1, 12, 24, 48, np.inf]
tenure_labels = ["0-12个月", "13-24个月", "25-48个月", "48个月以上"]
cleaned_df["TenureGroup"] = pd.cut(cleaned_df["Tenure"], bins=tenure_bins, labels=tenure_labels)
cleaned_df["IsMobileLogin"] = cleaned_df["PreferredLoginDevice"].eq("Mobile Phone").astype(int)

outlier_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_report = pd.DataFrame([
    {"字段": col, **iqr_outlier_summary(cleaned_df[col])} for col in outlier_cols
])
display(outlier_report)


,字段,Q1,Q3,下限,上限,候选异常值数量
0,WarehouseToHome,9.00,20.00,-7.50,36.50,2
1,OrderCount,1.00,3.00,-2.00,6.00,703
2,CashbackAmount,145.77,196.39,69.84,272.33,438


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [9]:
business_rule_report = pd.DataFrame({
    "规则": ["使用时长小于0", "仓库距离小于0", "订单数小于或等于0", "返现金额小于0"],
    "不合规记录数": [
        int((cleaned_df["Tenure"] < 0).sum()),
        int((cleaned_df["WarehouseToHome"] < 0).sum()),
        int((cleaned_df["OrderCount"] <= 0).sum()),
        int((cleaned_df["CashbackAmount"] < 0).sum()),
    ]
})
display(business_rule_report)
print("处理结论：业务规则不合规值仅记录并提交复核，不在缺少业务依据时自动删除。")


,规则,不合规记录数
0,使用时长小于0,0
1,仓库距离小于0,0
2,订单数小于或等于0,0
3,返现金额小于0,0


处理结论：业务规则不合规值仅记录并提交复核，不在缺少业务依据时自动删除。


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [10]:
quality_after = build_quality_report(cleaned_df)

assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique()
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique()
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique()
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns)
assert raw_df.shape == (5630, 20)
assert cleaned_df.shape == (5630, 22)

deliverables = {
    "data_quality_before.csv": quality_before,
    "data_quality_after.csv": quality_after,
    "cleaning_log.csv": cleaning_log,
    "ecommerce_customer_cleaned.csv": cleaned_df,
    "outlier_report.csv": outlier_report,
    "business_rule_report.csv": business_rule_report,
}
for filename, table in deliverables.items():
    index = filename.startswith("data_quality_")
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=index, encoding="utf-8-sig")
    assert path.exists()
    print("已输出：", path)

print("清洗前缺失总数：", int(raw_df.isna().sum().sum()))
print("清洗后缺失总数：", int(cleaned_df.isna().sum().sum()))
print("最终验收通过。")


已输出： c:\Users\Li Xiaofeng\Documents\muc-commerce-2-24012416\output\day04_project\data_quality_before.csv
已输出： c:\Users\Li Xiaofeng\Documents\muc-commerce-2-24012416\output\day04_project\data_quality_after.csv
已输出： c:\Users\Li Xiaofeng\Documents\muc-commerce-2-24012416\output\day04_project\cleaning_log.csv
已输出： c:\Users\Li Xiaofeng\Documents\muc-commerce-2-24012416\output\day04_project\ecommerce_customer_cleaned.csv
已输出： c:\Users\Li Xiaofeng\Documents\muc-commerce-2-24012416\output\day04_project\outlier_report.csv
已输出： c:\Users\Li Xiaofeng\Documents\muc-commerce-2-24012416\output\day04_project\business_rule_report.csv
清洗前缺失总数： 1856
清洗后缺失总数： 0
最终验收通过。


## 项目复盘（已完成）

本项目发现7个数值字段存在缺失值，登录设备、支付方式和偏好品类存在同义类别不一致，并发现若干IQR候选异常值。数值缺失采用总体中位数填补；同义类别按业务规则统一；候选异常值及业务不合规值只记录复核，不盲目删除。清洗后核心数值字段无缺失、类别口径统一，并新增生命周期分组和移动端标记，因此可作为第五天用户分析输入。IQR阈值、订单数非正值的处理方式，以及类别映射规则仍需业务人员确认。
